# Week 1: micrograd exercises

Do this **after** you finish Karpathy's micrograd video. About 3 hours total.

| Part | What you do | Time |
|---|---|---|
| 1 | Rebuild the core `Value` engine (`+`, `*`, `tanh`, `backward`) | 45 min |
| 2 | Break it on purpose (the `+=` bug) | 10 min |
| 3 | Extend your engine: `relu` + a gradient checker | 45 min |
| 4 | Train a neural net on real data with your engine | 45 min |
| 5 | Write it up for GitHub | 15 min |

**How to use this notebook**
- Fill in every cell marked `# TODO`. Then run the test cell under it: ✅ means correct, ❌ means not yet. An error usually means a method above it still returns nothing.
- Hints are hidden under **"Hint"**. Try for 10 minutes before opening one.
- **Rules:** don't look at Karpathy's code for Parts 1–3. Looking up Python syntax (or asking AI about syntax) is fine. The gradient math must be yours.
- If you peek, write down what you peeked at. That list is your review list.

Runs on Colab as-is (PyTorch and scikit-learn are preinstalled).

In [2]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import make_moons

random.seed(1337)
np.random.seed(1337)

def check(name, got, expected, tol=1e-6):
    # Prints ✅ / ❌ comparing your number to the expected one.
    if got is None:
        print(f"❌ {name}: got None (did you forget to return something?)")
        return False
    ok = abs(got - expected) < tol
    print(f"{'✅' if ok else '❌'} {name}: got {got:.6f}, expected {expected:.6f}")
    return ok

print("Setup done.")

Setup done.


---
## Part 1: Rebuild the core engine (45 min)

Fill in the `Value` class. Each operation must do **three things**:
1. Compute the output value (the forward pass).
2. Define a `_backward` function that pushes `out.grad` back into the inputs' `.grad` using the chain rule: **local derivative × out.grad**.
3. Attach that function to `out` (store it, don't call it) and return `out`.

`__add__` is half done as a model. The helpers at the bottom (`__neg__`, `__sub__`, `__radd__`, `__rmul__`) are given. They add no new ideas; they just let you write `2 * x` or `sum(...)`.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        # TODO: define _backward() so it adds the right amount to self.grad and other.grad
        #       then attach it to out
        def _backward():
            self.grad = out.grad * 1.0
            other.grad = out.grad * 1.0

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO: create out, define _backward, attach it, return out
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward
        return out

    def tanh(self):
        # TODO: forward: t = tanh(self.data)
        #       local derivative of tanh(x) is 1 - t**2
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out
        

    def backward(self):
        # TODO:
        # 1. Build a list `topo` of all nodes in the graph, where every node
        #    comes AFTER all of its children (_prev). Use a `visited` set.
        # 2. Set self.grad = 1.0
        # 3. Go through topo in REVERSE and call each node's _backward()
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward() #calling the backward function for each node in reverse topological order
    # ---- provided helpers ----
    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other   # makes 2 + x and sum() work
    def __rmul__(self, other): return self * other   # makes 2 * x work

<details><summary><b>Hint: __add__ and __mul__</b></summary>

- For `out = a + b`: the local derivative with respect to each input is **1**, so each input gets `1.0 * out.grad`.
- For `out = a * b`: the local derivative with respect to `a` is `b.data`, and with respect to `b` it's `a.data`.
- Use `+=`, not `=` (Part 2 shows why).
- Attach with `out._backward = _backward` (no parentheses!).
</details>

<details><summary><b>Hint: tanh</b></summary>

`math.tanh(x)` gives the forward value. `tanh` has only one input, so `_children` is `(self,)`.
</details>

<details><summary><b>Hint: backward / topological sort</b></summary>

Write a small recursive function `build(v)`: if `v` hasn't been visited, mark it visited, call `build` on each child in `v._prev`, then append `v` to `topo`. Call `build(self)`.
</details>

In [4]:
# ---- Test Part 1: your engine vs. PyTorch ----
def torch_val(x):
    return torch.tensor(x, dtype=torch.double, requires_grad=True)

# Test A: simple expression
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
e = (a * b + c).tanh()
e.backward()
ta, tb, tc = torch_val(2.0), torch_val(-3.0), torch_val(10.0)
te = (ta * tb + tc).tanh(); te.backward()
print("Test A: (a*b + c).tanh()")
check("  output", e.data, te.item())
check("  a.grad", a.grad, ta.grad.item())
check("  b.grad", b.grad, tb.grad.item())
check("  c.grad", c.grad, tc.grad.item())

# Test B: a variable used more than once, plus constants
x = Value(-1.5)
y = x * x * 2 + x * 3 - 4
y.backward()
tx = torch_val(-1.5)
ty = tx * tx * 2 + tx * 3 - 4; ty.backward()
print("Test B: x*x*2 + x*3 - 4")
check("  output", y.data, ty.item())
check("  x.grad", x.grad, tx.grad.item())

Test A: (a*b + c).tanh()
✅   output: got 0.999329, expected 0.999329
✅   a.grad: got -0.004023, expected -0.004023
✅   b.grad: got 0.002682, expected 0.002682
✅   c.grad: got 0.001341, expected 0.001341
Test B: x*x*2 + x*3 - 4
✅   output: got -4.000000, expected -4.000000
✅   x.grad: got -3.000000, expected -3.000000


True

---
## Part 2: Break it on purpose (10 min)

**Before running anything**, answer in the cell below:
1. For `b = a + a` with `a = 3`, what should `a.grad` be after `b.backward()`? Why?
2. If `__add__`'s `_backward` used `=` instead of `+=`, what would `a.grad` be instead?

In [11]:
# TODO: write your predictions as comments
a = 3
b = a + a 
print(f"b = {b}")  # b = 6
h = 0.001
b_new = (a+h) + (a+h)
print(f"b_new = {b_new}")  # b_new = 6.001

b_dev = (b_new - b) / h 
print(f"b_dev = {b_dev}")  # b_dev = 1

b = 6
b_new = 6.002
b_dev = 1.9999999999997797


**Now test it:**
1. Run the cell below with your correct engine.
2. Go back to Part 1, change `+=` to `=` in `__add__`'s `_backward`, re-run the class cell, then re-run the cell below.
3. **Change it back** to `+=` and re-run the class cell before continuing.

In [10]:
a = Value(3.0)
b = a + a
b.backward()
print("a.grad =", a.grad)

a.grad = 2.0


---
## Part 3: Extend your engine (45 min)

### 3a. Add `relu`

`relu(x) = max(0, x)`. Work out its local derivative yourself first: what is it when `x > 0`? When `x < 0`?

The last line attaches your function to the class as a method. It's the same idea as `out._backward = _backward`: storing a function on something so it can be called later.

In [ ]:
def relu(self):
    # TODO: forward value, then _backward, then attach and return out
    pass

Value.relu = relu   # attach as a method, so x.relu() works

<details><summary><b>Hint: relu</b></summary>

The derivative is 1 where the output is positive and 0 otherwise. In Python, `(out.data > 0)` is `True`/`False`, which multiplies like 1/0.
</details>

In [ ]:
# ---- Test 3a ----
p, q = Value(2.0), Value(-4.0)
r = (p * q).relu() + (p + q * -1).relu()
r.backward()
tp, tq = torch_val(2.0), torch_val(-4.0)
tr = (tp * tq).relu() + (tp + tq * -1).relu(); tr.backward()
check("relu output", r.data, tr.item())
check("p.grad", p.grad, tp.grad.item())
check("q.grad", q.grad, tq.grad.item())

### 3b. Write a gradient checker

You won't always have PyTorch to compare against, but you can always check gradients **numerically**: nudge one input up and down by a tiny `h` and see how the output moves.

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x_i + h) - f(x_i - h)}{2h}$$

Write `numerical_grad`. `grad_check` (given) uses it to compare against your backprop.

In [ ]:
def numerical_grad(f, inputs, h=1e-5):
    # f: a function that takes a list of floats and returns a float
    # inputs: a list of floats
    # returns: a list with one numerical gradient per input
    # TODO
    pass


def grad_check(build_expr, inputs):
    # Given: compares backprop gradients to numerical ones.
    vals = [Value(x) for x in inputs]
    out = build_expr(vals)
    out.backward()
    num = numerical_grad(lambda xs: build_expr([Value(x) for x in xs]).data, inputs)
    if num is None:
        print("❌ numerical_grad returned None")
        return
    for i, (v, n) in enumerate(zip(vals, num)):
        check(f"  input {i}", v.grad, n, tol=1e-4)

<details><summary><b>Hint: numerical_grad</b></summary>

For each index `i`: make a copy of `inputs` with element `i` increased by `h`, another copy with it decreased by `h`, call `f` on both, and apply the formula. `list(inputs)` makes a copy.
</details>

In [ ]:
# ---- Test 3b ----
print("Expression 1: x*y + y.tanh()")
grad_check(lambda v: v[0] * v[1] + v[1].tanh(), [0.5, -1.2])

print("Expression 2: a small neuron, relu(w1*x1 + w2*x2 + b)")
grad_check(lambda v: (v[0] * v[1] + v[2] * v[3] + v[4]).relu(), [0.8, 1.5, -0.3, 2.0, 0.1])

# TODO (optional): invent your own expression and check it

**Think about it:** expression 2 at an input where the relu's input is exactly 0 would make the numerical and backprop gradients disagree. Why? (Write your answer in your notes.)

---
## Part 4: Train a neural net with your engine (45 min)

The `Neuron`, `Layer` and `MLP` classes are given; they contain no new ideas beyond the video. They use **your** `relu`.

Your job: the **training loop**.

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
# The data: two interleaving half-moons. Labels are -1 and +1.
X_np, y_np = make_moons(n_samples=100, noise=0.1, random_state=1)
X = X_np.tolist()
y = [int(v) * 2 - 1 for v in y_np]

plt.figure(figsize=(5, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y, s=20, cmap='jet')
plt.title("make_moons: can a straight line separate these?")
plt.show()

**The loss (given).** This is a *hinge loss*: for each point, the score should have the same sign as the label, with a margin of at least 1. A point is penalized by `relu(1 - y·score)`, so correct and confident points contribute 0. A small L2 term keeps the weights from growing huge. (You'll study where losses like this come from in Phase 2.)

In [ ]:
def loss_fn(model):
    scores = [model(x) for x in X]
    data_loss = sum((1 + -yi * si).relu() for yi, si in zip(y, scores)) * (1.0 / len(y))
    reg_loss = 1e-4 * sum(p * p for p in model.parameters())
    accuracy = sum((yi > 0) == (si.data > 0) for yi, si in zip(y, scores)) / len(y)
    return data_loss + reg_loss, accuracy

**Your training loop.** Each step has four parts. The order matters, so think about why before you write it.

In [ ]:
random.seed(1337)
model = MLP(2, [16, 16, 1])
print("number of parameters:", len(model.parameters()))

history = []
steps = 50
for step in range(steps):
    learning_rate = 1.0 - 0.9 * step / steps   # decays from 1.0 to 0.1

    # TODO 1: forward pass, i.e. get `loss` and `acc` from loss_fn(model)

    # TODO 2: zero every parameter's gradient

    # TODO 3: backward pass

    # TODO 4: update every parameter: move it a small step against its gradient

    history.append(loss.data)
    if step % 5 == 0 or step == steps - 1:
        print(f"step {step:2d}  loss {loss.data:.4f}  accuracy {acc*100:.0f}%")

<details><summary><b>Hint: training loop</b></summary>

- Step 2 sets `p.grad = 0.0` for every `p` in `model.parameters()`. Why can't you skip it? (Think about `+=`.)
- Step 4 is `p.data -= learning_rate * p.grad`. Why minus?

Target: accuracy should reach **about 95–100%** within 50 steps (the loss can bounce a little; that is normal). Each step takes a second or two, since this is pure Python.
</details>

In [ ]:
# ---- Plots (given) ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("Training loss")

h = 0.25
x_min, x_max = X_np[:, 0].min() - 1, X_np[:, 0].max() + 1
y_min, y_max = X_np[:, 1].min() - 1, X_np[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()].tolist()
Z = np.array([model(pt).data > 0 for pt in grid]).reshape(xx.shape)

axes[1].contourf(xx, yy, Z, cmap='Spectral', alpha=0.8)
axes[1].scatter(X_np[:, 0], X_np[:, 1], c=y, s=20, cmap='jet', edgecolors='k')
axes[1].set_title("Decision boundary learned by your engine")
plt.tight_layout()
plt.savefig("micrograd_moons.png", dpi=120)
plt.show()

**Experiments (if you have time).** Change one thing, predict what happens, then run:
- Remove the zero-grad step. What happens to the loss?
- Use a fixed learning rate of 0.01, then 3.0.
- Try `MLP(2, [4, 1])`, a much smaller network. Can it still separate the moons?

---
## Part 5: Write it up (15 min)

Push this notebook and `micrograd_moons.png` to your `scratch-ml` repo. Then write a short README section answering, **in your own words**:

1. What does `_backward` do, and why is it stored on `out` instead of being called right away?
2. Why does the backward pass need a topological order?
3. Why `+=` and not `=`? (Use your Part 2 result.)
4. What did the decision-boundary plot show, and what happened in your experiments?

**Also note:** what did you have to peek at? That's your review list for next week.